In [1]:
from pyspark.sql import functions as F

# Load Gold tables
dim_date = spark.table("DimDate")
dim_zone = spark.table("DimZone")
fact_taxi = spark.table("FactTaxiDaily")

print("========== DIMDATE AUDIT ==========")

print("Total rows:", dim_date.count())
print("Distinct dates:", dim_date.select("date").distinct().count())

display(
    dim_date.select(
        F.min("date").alias("Minimum Date"),
        F.max("date").alias("Maximum Date")
    )
)

print("========== DIMZONE AUDIT ==========")

print("Total rows:", dim_zone.count())
print(
    "Distinct Location IDs:",
    dim_zone.select("location_id").distinct().count()
)

print("Duplicate Location IDs:")

display(
    dim_zone.groupBy("location_id")
    .count()
    .filter(F.col("count") > 1)
)

print("========== UNKNOWN ZONE AUDIT ==========")

zone_totals = (
    fact_taxi
    .groupBy("pickup_location_id")
    .agg(
        F.sum("total_trips").alias("total_trips"),
        F.sum("total_passengers").alias("total_passengers"),
        F.sum("total_revenue").alias("total_revenue")
    )
)

unknown_zones = zone_totals.join(
    dim_zone.select("location_id"),
    zone_totals.pickup_location_id == dim_zone.location_id,
    "left_anti"
)

display(unknown_zones.orderBy("pickup_location_id"))

print("========== AUDIT COMPLETED ==========")

StatementMeta(, 128a245b-4b99-4e46-a7f8-2f20ca3ced3c, 3, Finished, Available, Finished, False)

========== DIMDATE AUDIT ==========
Total rows: 33
Distinct dates: 33


SynapseWidget(Synapse.DataFrame, cf7cfa6e-9016-4899-869b-328873aefed6)

========== DIMZONE AUDIT ==========
Total rows: 262
Distinct Location IDs: 262
Duplicate Location IDs:


SynapseWidget(Synapse.DataFrame, 1aa5e23f-66c7-470b-a884-51534351baf1)

========== UNKNOWN ZONE AUDIT ==========


SynapseWidget(Synapse.DataFrame, aaef884a-dbf1-4983-aa1a-caa9a0c2c47e)

========== AUDIT COMPLETED ==========


In [3]:
from pyspark.sql import functions as F

taxi = spark.table("FactTaxiDaily")
air = spark.table("FactAirQualityDaily")

print("===== TAXI EXTRA DATES =====")

display(
    taxi
    .filter(
        (F.col("date") < F.lit("2026-01-01")) |
        (F.col("date") > F.lit("2026-01-31"))
    )
    .groupBy("date")
    .agg(
        F.sum("total_trips").alias("trips"),
        F.sum("total_revenue").alias("revenue")
    )
    .orderBy("date")
)

print("===== AIR QUALITY EXTRA DATES =====")

display(
    air
    .filter(
        (F.col("date") < F.lit("2026-01-01")) |
        (F.col("date") > F.lit("2026-01-31"))
    )
    .orderBy("date")
)

print("===== TAXI LOCATION ID CHECK =====")

display(
    taxi.select("pickup_location_id")
    .distinct()
    .orderBy("pickup_location_id")
)

StatementMeta(, 128a245b-4b99-4e46-a7f8-2f20ca3ced3c, 5, Finished, Available, Finished, False)

===== TAXI EXTRA DATES =====


SynapseWidget(Synapse.DataFrame, 94e1c118-28da-4d27-9729-142f5932951d)

===== AIR QUALITY EXTRA DATES =====


SynapseWidget(Synapse.DataFrame, 36391fc4-07ba-4e13-8a9c-bafd76c05aab)

===== TAXI LOCATION ID CHECK =====


SynapseWidget(Synapse.DataFrame, befab900-6842-4330-ad56-64b39aaf080a)

In [1]:
from pyspark.sql import functions as F

taxi_silver = spark.read.table("NYC_Silver_LH.dbo.yellow_taxi_clean")

print("===== SILVER TAXI TABLE COLUMNS =====")
print(taxi_silver.columns)

print("===== EXTRA DATE RECORDS =====")

display(
    taxi_silver
    .filter(
        (F.col("pickup_date") < F.lit("2026-01-01")) |
        (F.col("pickup_date") > F.lit("2026-01-31"))
    )
    .select(
        "pickup_datetime",
        "dropoff_datetime",
        "pickup_date",
        "trip_distance",
        "fare_amount"
    )
    .orderBy("pickup_datetime")
)

StatementMeta(, 8974dd75-70d7-42fc-abb4-70fc25ebaf15, 3, Finished, Available, Finished, False)

{"ts": "2026-11-05 17:06:09.911", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `pickup_datetime` cannot be resolved. Did you mean one of the following? [`pickup_date`, `pickup_day`, `tpep_pickup_datetime`, `pickup_hour`, `pulocationid`]. SQLSTATE: 42703", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o6147.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `pickup_datetime` cannot be resolved. Did you mean one of the following? [`pickup_date`, `pickup_day`, `tpep_pickup_datetime`, `pickup_hour`, `pulocationid`]. SQLSTATE: 42703;\n'Project ['pickup_datetime, 'dropoff_datetime, 

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `pickup_datetime` cannot be resolved. Did you mean one of the following? [`pickup_date`, `pickup_day`, `tpep_pickup_datetime`, `pickup_hour`, `pulocationid`]. SQLSTATE: 42703;
'Project ['pickup_datetime, 'dropoff_datetime, pickup_date#413, trip_distance#397, fare_amount#403]
+- Filter ((pickup_date#413 < cast(2026-01-01 as date)) OR (pickup_date#413 > cast(2026-01-31 as date)))
   +- SubqueryAlias spark_catalog.chimcobldhq2ajip8dfkqrr2d5m6it3pbt0msobcf5q6iorj4l75igqvadkmotj5e9fkoi15chh6u.yellow_taxi_clean
      +- Relation spark_catalog.chimcobldhq2ajip8dfkqrr2d5m6it3pbt0msobcf5q6iorj4l75igqvadkmotj5e9fkoi15chh6u.yellow_taxi_clean[vendorid#393,tpep_pickup_datetime#394,tpep_dropoff_datetime#395,passenger_count#396L,trip_distance#397,ratecodeid#398L,store_and_fwd_flag#399,pulocationid#400,dolocationid#401,payment_type#402L,fare_amount#403,extra#404,mta_tax#405,tip_amount#406,tolls_a

In [2]:
from pyspark.sql import functions as F

taxi_silver = spark.read.table("NYC_Silver_LH.dbo.yellow_taxi_clean")

display(
    taxi_silver
    .filter(
        (F.col("pickup_date") < F.lit("2026-01-01")) |
        (F.col("pickup_date") > F.lit("2026-01-31"))
    )
    .select(
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "pickup_date",
        "trip_distance",
        "fare_amount"
    )
    .orderBy("tpep_pickup_datetime")
)

StatementMeta(, 8974dd75-70d7-42fc-abb4-70fc25ebaf15, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8381a75a-45ef-4ba5-a613-9e31c3a4fc20)

In [3]:
from pyspark.sql import functions as F

taxi = spark.read.table("NYC_Silver_LH.dbo.yellow_taxi_clean")

print("===== DATA QUALITY SUMMARY =====")

display(
    taxi.agg(
        F.count("*").alias("total_rows"),
        F.sum(
            F.when(F.col("pickup_date") < "2026-01-01", 1).otherwise(0)
        ).alias("before_january"),
        F.sum(
            F.when(F.col("pickup_date") > "2026-01-31", 1).otherwise(0)
        ).alias("after_january"),
        F.sum(
            F.when(F.col("fare_amount") < 0, 1).otherwise(0)
        ).alias("negative_fares"),
        F.sum(
            F.when(F.col("trip_distance") <= 0, 1).otherwise(0)
        ).alias("nonpositive_distance")
    )
)

print("===== NEGATIVE FARE RECORDS =====")

display(
    taxi.filter(F.col("fare_amount") < 0)
    .select(
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_distance",
        "fare_amount",
        "total_amount",
        "pulocationid",
        "dolocationid",
        "payment_type"
    )
    .limit(20)
)

StatementMeta(, 8974dd75-70d7-42fc-abb4-70fc25ebaf15, 5, Finished, Available, Finished, False)

===== DATA QUALITY SUMMARY =====


SynapseWidget(Synapse.DataFrame, 7a8a4054-9eba-4ff4-8f1c-fe4877a01002)

===== NEGATIVE FARE RECORDS =====


SynapseWidget(Synapse.DataFrame, 9cbdc0c9-a9df-46bc-a9f9-4392750ae9db)

In [4]:
from pyspark.sql import functions as F

taxi = spark.read.table("NYC_Silver_LH.dbo.yellow_taxi_clean")

print("===== NEGATIVE FARE BY PAYMENT TYPE =====")

display(
    taxi
    .filter(F.col("fare_amount") < 0)
    .groupBy("payment_type")
    .agg(
        F.count("*").alias("negative_fare_rows"),
        F.sum("fare_amount").alias("fare_total"),
        F.sum("total_amount").alias("total_amount_sum")
    )
    .orderBy("payment_type")
)

print("===== NEGATIVE TOTAL AMOUNT SUMMARY =====")

display(
    taxi.agg(
        F.sum(
            F.when(F.col("total_amount") < 0, 1).otherwise(0)
        ).alias("negative_total_amount_rows"),
        F.sum("total_amount").alias("overall_total_amount"),
        F.sum(
            F.when(F.col("fare_amount") < 0, F.col("total_amount"))
            .otherwise(0)
        ).alias("total_amount_from_negative_fares")
    )
)

StatementMeta(, 8974dd75-70d7-42fc-abb4-70fc25ebaf15, 6, Finished, Available, Finished, False)

===== NEGATIVE FARE BY PAYMENT TYPE =====


SynapseWidget(Synapse.DataFrame, 888594cb-2c24-4d9c-83f7-34036aedc085)

===== NEGATIVE TOTAL AMOUNT SUMMARY =====


SynapseWidget(Synapse.DataFrame, 399b7927-99ca-4f4a-9b8c-70b6e7124ec2)

In [5]:
from pyspark.sql import functions as F

taxi = spark.read.table("NYC_Silver_LH.dbo.yellow_taxi_clean")

print("===== NEGATIVE TOTAL AMOUNT WITHOUT NEGATIVE FARE =====")

display(
    taxi
    .filter(
        (F.col("total_amount") < 0) &
        (F.col("fare_amount") >= 0)
    )
    .groupBy("payment_type")
    .agg(
        F.count("*").alias("row_count"),
        F.sum("fare_amount").alias("fare_sum"),
        F.sum("total_amount").alias("total_amount_sum")
    )
    .orderBy("payment_type")
)

print("===== SAMPLE RECORDS =====")

display(
    taxi
    .filter(
        (F.col("total_amount") < 0) &
        (F.col("fare_amount") >= 0)
    )
    .select(
        "tpep_pickup_datetime",
        "fare_amount",
        "total_amount",
        "payment_type",
        "pulocationid",
        "dolocationid"
    )
    .limit(15)
)

StatementMeta(, 8974dd75-70d7-42fc-abb4-70fc25ebaf15, 7, Finished, Available, Finished, False)

===== NEGATIVE TOTAL AMOUNT WITHOUT NEGATIVE FARE =====


SynapseWidget(Synapse.DataFrame, 0910cd5e-a54b-472f-86c4-34e57de6b94b)

===== SAMPLE RECORDS =====


SynapseWidget(Synapse.DataFrame, c776e087-596f-42ac-9380-a2dbf53858ee)

In [6]:
from pyspark.sql import functions as F

taxi = spark.read.table("NYC_Silver_LH.dbo.yellow_taxi_clean")

display(
    taxi.agg(
        F.count("*").alias("total_rows"),

        F.sum(
            F.when(F.col("fare_amount") < 0, 1).otherwise(0)
        ).alias("negative_fare"),

        F.sum(
            F.when(F.col("fare_amount") == 0, 1).otherwise(0)
        ).alias("zero_fare"),

        F.sum(
            F.when(F.col("total_amount") < 0, 1).otherwise(0)
        ).alias("negative_total_amount"),

        F.sum(
            F.when(
                (F.col("fare_amount") < 0) &
                (F.col("total_amount") < 0), 1
            ).otherwise(0)
        ).alias("both_negative"),

        F.sum(
            F.when(
                (F.col("fare_amount") >= 0) &
                (F.col("total_amount") < 0), 1
            ).otherwise(0)
        ).alias("negative_total_nonnegative_fare")
    )
)


StatementMeta(, 8974dd75-70d7-42fc-abb4-70fc25ebaf15, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 87758f2a-ca11-4ea2-adc3-896c803d71aa)

In [7]:
from pyspark.sql import functions as F

taxi = spark.read.table("NYC_Silver_LH.dbo.yellow_taxi_clean")

display(
    taxi.agg(
        F.sum("fare_amount").alias("sum_fare_amount"),
        F.sum("total_amount").alias("sum_total_amount"),

        F.sum(
            F.when(F.col("fare_amount") > 0, F.col("fare_amount"))
            .otherwise(0)
        ).alias("positive_fare_only"),

        F.sum(
            F.when(F.col("total_amount") > 0, F.col("total_amount"))
            .otherwise(0)
        ).alias("positive_total_only"),

        F.sum(
            F.when(F.col("total_amount") < 0, F.col("total_amount"))
            .otherwise(0)
        ).alias("negative_total_only")
    )
)

StatementMeta(, 8974dd75-70d7-42fc-abb4-70fc25ebaf15, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 38dbe9a4-c2f3-436a-b751-cb10bab5c137)

In [8]:
from pyspark.sql import functions as F

taxi = spark.read.table("NYC_Silver_LH.dbo.yellow_taxi_clean")

print("===== DUPLICATE TRIP AUDIT =====")

trip_key = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "pulocationid",
    "dolocationid",
    "trip_distance",
    "fare_amount",
    "total_amount"
]

duplicates = (
    taxi.groupBy(*trip_key)
    .count()
    .filter(F.col("count") > 1)
)

display(
    duplicates.agg(
        F.count("*").alias("duplicate_groups"),
        F.sum("count").alias("rows_in_duplicate_groups"),
        F.sum(F.col("count") - 1).alias("extra_duplicate_rows")
    )
)

print("===== SAMPLE DUPLICATES =====")

display(duplicates.orderBy(F.desc("count")).limit(10))

StatementMeta(, 8974dd75-70d7-42fc-abb4-70fc25ebaf15, 10, Finished, Available, Finished, False)

===== DUPLICATE TRIP AUDIT =====


SynapseWidget(Synapse.DataFrame, d29d6e24-5721-4ca3-b33b-a6e567da774f)

===== SAMPLE DUPLICATES =====


SynapseWidget(Synapse.DataFrame, 6b151f03-fa22-44bd-87e9-5114ccdba2fa)